# Thesis Figure 12 Ã¢â‚¬â€ Saliency Heatmap

For one experiment, one parent run, and a list of image names, produces per image:
- **`{image_name}_saliency`** Ã¢â‚¬â€ jet-colourmap saliency heatmap (aggregated across all lengths)
- **`{image_name}_regions_l{REGION_LENGTH}`** Ã¢â‚¬â€ red region overlay at the chosen length

The saliency is built from the CIAO regions of all child runs that share the same
**algorithm label + seed** as the given parent run, across all available lengths.

Edit only the **Configuration** cell.

In [1]:
import json
import os
import re
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
from mlflow.tracking import MlflowClient


REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from ciao.data.preprocessing import load_and_preprocess_image
from ciao.metrics import build_saliency_map
from ciao.visualization.visualization import _to_hwc


# Ã¢â€â‚¬Ã¢â€â‚¬ MLflow Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
TRACKING_URI = "https://mlflow.rationai.cloud.e-infra.cz/"
EXPERIMENT_NAME = "algorithm-comparison-imagenet-s"
os.environ["MLFLOW_TRACKING_USERNAME"] = "YOUR_MLFLOW_USERNAME"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "YOUR_MLFLOW_PASSWORD"

# Ã¢â€â‚¬Ã¢â€â‚¬ Run & images Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
PARENT_RUN_ID = (
    "bdc01c4c4a2840afa5b84da2160d9bc6"  # any parent run for the desired label+seed
)
IMAGES = [
    "imagenets_0125.jpg",
    "imagenets_0120.jpg",
    "imagenets_0065.jpg",
    "imagenets_0063.jpg",
    "imagenets_0061.jpg",
]
LOCAL_IMAGES_PATH = Path("REPO_ROOT/images/images")

# Ã¢â€â‚¬Ã¢â€â‚¬ Region overlay Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
REGION_LENGTH = 60  # desired length used for the region overlay output
REGION_ALPHA = 0.55  # opacity of the red region overlay

# Ã¢â€â‚¬Ã¢â€â‚¬ Saliency settings Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
SIGMA_FRACTION = 0.03  # Gaussian sigma = fraction of shorter image side
SAL_ALPHA = 0.55  # opacity of the jet saliency overlay

# Ã¢â€â‚¬Ã¢â€â‚¬ Output Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
OUTPUT_DIR = Path("output/12_saliency_heatmap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)
_name_re = re.compile(r"^(?P<label>.+)-(?P<seed>\d+)-length-(?P<length>\d+)$")


def _npy(run_id: str, name: str) -> np.ndarray | None:
    try:
        with tempfile.TemporaryDirectory() as d:
            p = client.download_artifacts(run_id, name, dst_path=d)
            return np.load(p)
    except Exception:
        return None


def _json_art(run_id: str, path: str) -> dict | None:
    try:
        with tempfile.TemporaryDirectory() as d:
            p = client.download_artifacts(run_id, path, dst_path=d)
            with open(p) as f:
                return json.load(f)
    except Exception:
        return None


def _binary_mask(segs: np.ndarray, ids: list[int]) -> np.ndarray:
    m = np.zeros(segs.shape, dtype=bool)
    for s in ids:
        m |= segs == s
    return m


def _region_segments(run_id: str) -> list[list[int]]:
    regions, idx = [], 0
    while True:
        d = _json_art(run_id, f"region_{idx}/segments.json")
        if d is None:
            break
        regions.append(d["segments"])
        idx += 1
    return regions


def _red_overlay(
    img: np.ndarray, mask: np.ndarray, alpha: float = REGION_ALPHA
) -> np.ndarray:
    out = img.copy()
    out[mask] = out[mask] * (1 - alpha) + np.array([1.0, 0.0, 0.0]) * alpha
    return out


def _save_single(img_np: np.ndarray, path: Path, dpi: int = 150) -> None:
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img_np, interpolation="nearest")
    ax.axis("off")
    fig.tight_layout(pad=0)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", pad_inches=0)
    fig.savefig(path.with_suffix(".pdf"), dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


def _save_saliency(
    img_np: np.ndarray, sal: np.ndarray, path: Path, dpi: int = 150
) -> None:
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img_np, interpolation="nearest")
    ax.imshow(sal, cmap="jet", alpha=SAL_ALPHA, vmin=0.0, vmax=1.0)
    ax.axis("off")
    fig.tight_layout(pad=0)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", pad_inches=0)
    fig.savefig(path.with_suffix(".pdf"), dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

In [3]:
# Resolve label+seed from the given parent run, then build a full lookup
# {image_name: {length: child_run_id}} for all sibling parent runs.

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    raise RuntimeError(f"Experiment '{EXPERIMENT_NAME}' not found")

parent_run = client.get_run(PARENT_RUN_ID)
parent_name = parent_run.data.tags.get("mlflow.runName", "")
m = _name_re.match(parent_name)
if not m:
    raise RuntimeError(
        f"Parent run name '{parent_name}' does not match '<label>-<seed>-length-<n>' pattern"
    )
LABEL = m.group("label")
SEED = m.group("seed")
print(f"Parent run  : {parent_name}")
print(f"Label       : {LABEL}")
print(f"Seed        : {SEED}")

all_runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id], output_format="pandas"
)
all_parents = all_runs[all_runs["tags.mlflow.parentRunId"].isna()].copy()
all_children = all_runs[all_runs["tags.mlflow.parentRunId"].notna()].copy()


def _parse(name):
    mm = _name_re.match(str(name) if isinstance(name, str) else "")
    if not mm:
        return None, None, None
    return mm.group("label"), mm.group("seed"), int(mm.group("length"))


_p = all_parents["tags.mlflow.runName"].apply(_parse)
all_parents = all_parents.assign(
    label=_p.apply(lambda x: x[0]),
    seed=_p.apply(lambda x: x[1]),
    desired_length=_p.apply(lambda x: x[2]),
)

sibling_parents = all_parents[
    (all_parents["label"] == LABEL) & (all_parents["seed"] == SEED)
].copy()
print(f"Sibling parent runs : {len(sibling_parents)}")
print(f"Lengths             : {sorted(sibling_parents['desired_length'].tolist())}")

# {image_name: {length: child_run_id}}
merged = all_children.merge(
    sibling_parents[["run_id", "desired_length"]],
    left_on="tags.mlflow.parentRunId",
    right_on="run_id",
    suffixes=("", "_parent"),
    how="inner",
)
merged["image_name"] = merged["tags.mlflow.runName"]

img_lookup: dict[str, dict[int, str]] = {}
for _, r in merged.iterrows():
    img_lookup.setdefault(r["image_name"], {})[int(r["desired_length"])] = r["run_id"]

for img in IMAGES:
    lengths = sorted(img_lookup.get(img, {}).keys())
    print(f"  {img}: lengths = {lengths}")

Parent run  : ucb-42-length-60
Label       : ucb
Seed        : 42
Sibling parent runs : 4
Lengths             : [15, 30, 60, 90]
  imagenets_0125.jpg: lengths = [15, 30, 60, 90]
  imagenets_0120.jpg: lengths = [15, 30, 60, 90]
  imagenets_0065.jpg: lengths = [15, 30, 60, 90]
  imagenets_0063.jpg: lengths = [15, 30, 60, 90]
  imagenets_0061.jpg: lengths = [15, 30, 60, 90]


In [4]:
device = torch.device("cpu")

for image_name in IMAGES:
    print(f"\n{'Ã¢â€â‚¬' * 60}")
    print(f"Processing: {image_name}")
    stem = Path(image_name).stem

    length_to_run = img_lookup.get(image_name, {})
    if not length_to_run:
        print("  No child runs found Ã¢â‚¬â€ skipping")
        continue

    img_tensor = load_and_preprocess_image(
        LOCAL_IMAGES_PATH / image_name, device=device
    )
    img_np = _to_hwc(img_tensor.unsqueeze(0))

    # Ã¢â€â‚¬Ã¢â€â‚¬ Saliency: collect binary masks across all lengths Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    binary_masks: list[np.ndarray] = []
    for length in sorted(length_to_run.keys()):
        rid = length_to_run[length]
        segs = _npy(rid, "segments.npy")
        if segs is None:
            print(f"  length={length}: segments.npy missing Ã¢â‚¬â€ skipping")
            continue
        reg_segs = _region_segments(rid)
        if not reg_segs:
            print(f"  length={length}: no region segments Ã¢â‚¬â€ skipping")
            continue
        all_ids = [s for reg in reg_segs for s in reg]
        binary_masks.append(_binary_mask(segs, all_ids))
        print(f"  length={length}: {len(reg_segs)} region(s), {len(all_ids)} segments")

    if binary_masks:
        saliency = build_saliency_map(binary_masks, sigma_fraction=SIGMA_FRACTION)
        _save_saliency(img_np, saliency, OUTPUT_DIR / f"{stem}_saliency.png")
        print(f"  Saved saliency Ã¢â€ â€™ {stem}_saliency.png / .pdf")
    else:
        print("  No masks built Ã¢â‚¬â€ saliency skipped")

    # Ã¢â€â‚¬Ã¢â€â‚¬ Region overlay at REGION_LENGTH Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
    rid_region = length_to_run.get(REGION_LENGTH)
    if rid_region is None:
        print(f"  No run for length={REGION_LENGTH} Ã¢â‚¬â€ region overlay skipped")
    else:
        segs = _npy(rid_region, "segments.npy")
        reg_segs = _region_segments(rid_region)
        if segs is not None and reg_segs:
            all_ids = [s for reg in reg_segs for s in reg]
            overlay = _red_overlay(img_np, _binary_mask(segs, all_ids))
            _save_single(overlay, OUTPUT_DIR / f"{stem}_regions_l{REGION_LENGTH}.png")
            print(
                f"  Saved regions  Ã¢â€ â€™ {stem}_regions_l{REGION_LENGTH}.png / .pdf"
            )
        else:
            print(
                f"  Artifacts missing for length={REGION_LENGTH} Ã¢â‚¬â€ region overlay skipped"
            )

print("\nDone.")


Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
Processing: imagenets_0125.jpg


  length=15: 1 region(s), 15 segments


  length=30: 1 region(s), 30 segments


  length=60: 1 region(s), 60 segments


  length=90: 1 region(s), 90 segments
  Saved saliency Ã¢â€ â€™ imagenets_0125_saliency.png / .pdf


...

  length=15: 1 region(s), 15 segments


  length=30: 1 region(s), 30 segments


  length=60: 1 region(s), 60 segments


  length=90: 1 region(s), 90 segments
  Saved saliency Ã¢â€ â€™ imagenets_0061_saliency.png / .pdf


  Saved regions  Ã¢â€ â€™ imagenets_0061_regions_l60.png / .pdf

Done.
